#Reading Data From Bronze

## Reading Prod table


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import col
from pyspark.sql.types import DateType

In [0]:
df_prod = spark.table('workspace.bronze.crm_prod_info')

In [0]:
display(df_prod)

#Data Transformation

## Transforming CRM Prod table
- Trim the string values
- column names are not friendly
- Date Casting
- Product Line Normalization
- Cost Cleanup
- Product Key Parsing

## Trimming 

In [0]:
for field in df_prod.schema.fields:
    if field.dataType == StringType():
        print(field.name)
        df_prod = df_prod.withColumn(field.name, trim(col(field.name)))

In [0]:
display(df_prod)

## Changing the column name to more user friendly names by creating a dict

In [0]:
#fetching the column names
prod_cols = df_prod.columns
prod_cols

In [0]:
rename_map = {
    'prd_id' :'product_id',
    'prd_key': 'product_key',
    'prd_nm' : 'product_name',
    'prd_cost': 'product_cost',
    'prd_line': 'product_line',
    'prd_start_dt': 'product_start_date',
    'prd_end_dt': 'product_end_date',
}

In [0]:
for old_val, new_val in rename_map.items():
    df_prod = df_prod.withColumnRenamed(old_val, new_val)
display(df_prod.limit(3))

## Date casting

In [0]:
df_prod = df_prod.withColumn("product_start_date", col("product_start_date").cast(DateType())) \
                 .withColumn("product_end_date", col("product_end_date").cast(DateType()))

display(df_prod.select("product_start_date", "product_end_date"))

##Product Line Normalization

In [0]:
#Getting the unique/distinct value from product_line
df_prod.select("product_line").distinct().show()

In [0]:
df_prod.select(count(when(col("product_line").isNull(), 'product_line')).alias("product_line")).show()

In [0]:
#normalizing the product line values, M - Mountains, R-Roads etc

df_prod = (
    df_prod
    .withColumn(
        "product_line",
        when(upper(col("product_line")) =="M","Mountain")
        .when(upper(col("product_line")) =="R","Road")
        .when(upper(col("product_line")) =="S","Other Sales")
        .when(upper(col("product_line")) =="T","Touring")
        .otherwise('n/a')
    )
)

In [0]:
#asserting the change
display(df_prod)

## Parsing the Product Key Column

In [0]:
df_prod = df_prod.withColumn("cat_id", regexp_replace(substring(col("product_key"), 1, 5), "-", "_"))
df_prod = df_prod.withColumn("product_key", substring(col("product_key"),7,length(col("product_key"))))

In [0]:
display(df_prod)

In [0]:
df_prod = df_prod.withColumn("product_cost", coalesce(col("Product_cost"), lit(0)))

In [0]:
display(df_prod)

#Writing to Silver 

In [0]:
df_prod.write.mode("overwrite").format("delta").saveAsTable('workspace.silver.crm_prod')

In [0]:
%sql
select * from workspace.silver.crm_prod limit 5;

In [0]:
%sql
select * from workspace.silver.crm_prod where product_key = 'BK-R93R-56'